In [1]:
%%bash
cat << 'EOF' > ~/tesi_graphrag/src/sidecar_manager.py
import sys
import json
import copy
import time
from pathlib import Path
from typing import Any, Dict, Optional, List

project_root = Path("~/tesi_graphrag").expanduser().resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
from src.config import DISTANCE_CLASS_FACTORS

class SidecarManager:
    """
    Istanziare una singola volta per filepath (.json).
    La stessa accortezza va realizzata non solo nello stesso notebook,
    ma anche tra notebook DIVERSI.
    Evitare cioè modifiche concorrenti, visto che non c'è nessun lock che le impedisca.
    Nello stesso notebook c'è un WARNING, ma tra notebook diversi non c'è avviso.
    """

    # Tiene traccia dei path per cui è stata creata un istanza.
    #  L'utilizzo di una cache infatti rende problematica la presenza di più istanze:
    #  ciascuna avrebbe una propria cache in RAM indipendente con il rischio che il loro contenuto
    #  diverga senza avvisi.
    #  Un implementazione singleton avrebbe complicato molto il codice, invece tramite
    #  questa accortezza se si istanzia una seconda istanza viene stampato un avviso (non bloccante)
    _seen_paths: set = set()

    DEFAULT_PALETTE = [
        "#4e79a7", "#f28e2b", "#e15759", "#76b7b2", "#59a14f",
        "#edc949", "#af7aa1", "#ff9da7", "#9c755f", "#bab0ab"
    ] # Colori di Default se non ne vengono assegnati altri

    def __init__(self, filepath="sidecar_edits.json"):
        # Inizializzazione di default; modificabile passandogli un diverso path come parametro
        self.filepath = (
            Path(filepath).expanduser().resolve()
            if filepath else Path("sidecar_edits.json").resolve()
        )

        # Verifica istanziazione singola
        if self.filepath in SidecarManager._seen_paths:
            print(
                f"[WARNING] Creata una seconda istanza di SidecarManager per lo stesso file "
                f"({self.filepath}). Ogni istanza mantiene una propria cache in RAM: se non e' "
                f"la stessa istanza ad essere condivisa e riutilizzata ovunque, le due cache "
                f"possono disallinearsi silenziosamente. Crea una sola istanza e passala "
                f"esplicitamente a tutti i componenti che leggono/scrivono il sidecar."
            )
        SidecarManager._seen_paths.add(self.filepath)

        # Crea la cache
        self._cache: Optional[Dict[str, Any]] = None

        self._ensure_file_exists()

    ### Accesso e gestione dati e cache

    def _ensure_file_exists(self):
        self.filepath.parent.mkdir(parents=True, exist_ok=True)
        if not self.filepath.exists():
            self.reset_all()

    @property
    def data(self) -> dict:
        """Accesso ai dati aggiornati forzando automaticamente il reload"""
        return self.load_data(force_reload=False)

    def load_data(self, force_reload: bool=False) -> dict:
        """
        Carica i dati del sidecaar.
        Sfrutta la cache in RAM per evitare I/O ripetuto e eventuale collo di bottiglia.
        Restituisce sempre la deepcopy, così eventuali mutazioni da parte del chiamante
        (prima di un save_data) non sporcano la cache_interna.
        """
        if self._cache is not None and not force_reload:
            return copy.deepcopy(self._cache)

        if self.filepath.exists():
            try:
                with open(self.filepath, "r", encoding="utf-8") as f:
                    self._cache = json.load(f)
                return copy.deepcopy(self._cache)
            except Exception as e:
                # Il file esiste ma non è leggibile (es. JSON corrotto per
                # un'interruzione a metà scrittura). Prima di scartarlo ne
                # salviamo una copia: la prossima save_data() sovrascriverà
                # il file originale, quindi senza backup il contenuto
                # eventualmente ancora recuperabile andrebbe perso per sempre.
                backup_path = self.filepath.with_suffix(f".corrotto.{int(time.time())}.json")
                try:
                    self.filepath.replace(backup_path)
                    print(f"<[WARNING]> File sidecar illeggibile ({e}). "
                          f"Backup del file originale salvato in: {backup_path}")
                except Exception:
                    print(f"<[WARNING]> File sidecar illeggibile ({e}) e impossibile crearne un backup.")


        default_structure = {
            "modified_adjacencies": {},
            "tag_overrides": {},
            "global_tags": {},
            "predetermined_queries": {}
        }
        self._cache = default_structure
        return copy.deepcopy(self._cache)

    def save_data(self, data: dict):
        """Salva il dizionario su disco in modo atomico e aggiorna la cache in RAM."""
        self.filepath.parent.mkdir(parents=True, exist_ok=True)

        # Scrittura atomica: si scrive prima su un file temporaneo e poi lo
        # si sostituisce al file definitivo. Path.replace (os.replace) e'
        # atomico su POSIX, quindi anche un'interruzione a meta' (crash,
        # restart del kernel) non puo' lasciare sidecar_edits.json in uno
        # stato troncato/corrotto: resta o la vecchia versione, o gia'
        # completa quella nuova.
        tmp_path = self.filepath.with_suffix(self.filepath.suffix + ".tmp")
        with open(tmp_path, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=2)
        tmp_path.replace(self.filepath)

        self._cache = copy.deepcopy(data)

    def invalidate_cache(self):
        """Forza la rilettura da filesystem alla successiva chiamata di load_data()."""
        self._cache = None

    def release_path(self) -> None:
        """
        Rimuove il path di questa istanza dal registro delle istanze viste.
        Da chiamare quando l'istanza non verrà più usata (es. in PreTagger.close()),
        così una nuova SidecarManager sullo stesso file non genera un falso avviso
        di "seconda istanza". NON funziona tra notebook diversi.
        """
        SidecarManager._seen_paths.discard(self.filepath)

    def get_global_tags(self) -> dict:
        """
        Restituisce il dizionario dei tag attivi, con contatore e colore corrente.
        """
        data = self.load_data()
        return data.get("global_tags", {})

    ### Manipolazione Classi di Distanza
    def save_pairwise_class_edits(self, chunk_id_1: str, chunk_id_2: str, distance_class: Optional[str]):
        """
        Salva o aggiorna la classe di distanza tra una coppia di chunk.
        Garantisce simmetria (chunk_1 <-> chunk_2) e pulizia automatica se la classe
        dovesse essere "INVARIATI" o non presente in DISTANCE_CLASS_FACTORS.
        Reindirizza automaticamente a save_predetermined_query_adjacency se uno degli ID riguarda una query pre-determinata
        """
        c1, c2 = str(chunk_id_1), str(chunk_id_2)
        if c1 == c2:
            return

        # Instradamento automatico verso le query pre-determinate
        if c1.startswith("predq_") or c2.startswith("predq_"):
            self.save_predetermined_query_adjacency(c1, c2, distance_class)
            return

        data = self.load_data()
        adj = data.setdefault("modified_adjacencies", {})

        norm_class = str(distance_class).upper().strip() if distance_class is not None else "INVARIATI"
        if norm_class not in DISTANCE_CLASS_FACTORS:
            norm_class = "INVARIATI"

        if norm_class == "INVARIATI":
            if c1 in adj and c2 in adj[c1]:
                del adj[c1][c2]
                if not adj[c1]:
                    del adj[c1]

            if c2 in adj and c1 in adj[c2]:
                del adj[c2][c1]
                if not adj[c2]:
                    del adj[c2]

            print(f"<<| Adiacenza rimossa (ripristino a INVARIATI) tra [{c1}] e [{c2}] |>>")
        else:
            adj.setdefault(c1, {})[c2] = {"distance_class": norm_class}
            adj.setdefault(c2, {})[c1] = {"distance_class": norm_class}
            print(f"<<| Modifica adiacenza salvata tra [{c1}] e [{c2}]: classe={norm_class} |>>")

        self.save_data(data)

    def save_pairwise_class_edits_batch(self, edits: List[Dict[str, Any]]):
        """
        Salva un blocco di modifiche adiacenze in un'unica operazione di I/O.
        Ogni voce di 'edits' deve contenere: 'chunk_1', 'chunk_2', 'distance_class'.
        Smista automaticamente gli elementi coinvolgenti 'predq_' al relativo batch dedicato.
        """
        if not edits or not isinstance(edits, list):
            return

        standard_edits = []
        predq_edits = []

        for edit in edits:
            if edit and "chunk_1" in edit and "chunk_2" in edit:
                c1, c2 = str(edit["chunk_1"]), str(edit["chunk_2"])
                if c1.startswith("predq_") or c2.startswith("predq_"):
                    predq_edits.append(edit)
                else:
                    standard_edits.append(edit)

        # Esegue prima l'eventuale sotto-batch per le query pre-determinate
        if predq_edits:
            self.save_predetermined_query_adjacencies_batch(predq_edits)

        if not standard_edits:
            return

        data = self.load_data()
        adj = data.setdefault("modified_adjacencies", {})
        updated = False

        for edit in standard_edits:
            c1, c2 = str(edit["chunk_1"]), str(edit["chunk_2"])
            if c1 == c2:
                continue
            d_cls = edit.get("distance_class")
            norm_class = str(d_cls).upper().strip() if d_cls is not None else "INVARIATI"

            if norm_class not in DISTANCE_CLASS_FACTORS:
                norm_class = "INVARIATI"

            if norm_class == "INVARIATI":
                if c1 in adj and c2 in adj[c1]:
                    del adj[c1][c2]
                    if not adj[c1]:
                        del adj[c1]
                if c2 in adj and c1 in adj[c2]:
                    del adj[c2][c1]
                    if not adj[c2]:
                        del adj[c2]
            else:
                adj.setdefault(c1, {})[c2] = {"distance_class": norm_class}
                adj.setdefault(c2, {})[c1] = {"distance_class": norm_class}
            updated = True

        if updated:
            self.save_data(data)
            print(f"<<| Batch adiacenze standard salvato: {len(standard_edits)} elementi processati |>>")
    # Estrazione adiacenze
    def get_chunk_adjacencies(self, chunk_id: str) -> Dict[str, Dict[str, str]]:
        """
        Restituisce il dizionario delle adiacenze modificate per un singolo chunk.
        es. return format: {'chunk_2': {'distance_class': 'AVVICINATI'}}
        """
        data = self.load_data()
        adj = data.get("modified_adjacencies", {})
        return adj.get(str(chunk_id), {})

    def get_all_modified_adjacencies(self) -> Dict[str, Dict[str, Dict[str, str]]]:
        """Restituisce l'intero dizionario delle adiacenze modificate."""
        data = self.load_data()
        return data.get("modified_adjacencies", {})

    ### Manipolazione TAG
    def add_tag_override(self, chunk_id: str, tag: str, color: Optional[str] = None):
        """
        Aggiunge un singolo tag a un chunk o a una query pre-determinata.
        Aggiorna global_tags SOLO per i chunk standard del database.
        """
        if not tag:
            return
        self.add_tag_override_for_chunk(chunk_id, [tag])
        
    def add_tag_override_for_chunk(self, chunk_id: str, tags: List[str]):
        """
        Assegna una lista di tag a un singolo chunk o a una query pre-determinata, aggiornando il registro globale
        ed eseguendo un uica operazione di scrittura su disco per l'intero chunk.
        Da utilizzare ad esempio per il pre-tagging, ottimizzando così l'operazione.
        Usato anche nel singolo tag_override per centralizzare l'operazione.
        Aggiorna global_tags SOLO per i chunk del database.
        """
        if not tags:
            return
    
        data = self.load_data()
        chunk_id = str(chunk_id)
    
        # Gestione Query Pre-Determinata (NON tocca global_tags)
        #  I tag graficamente aggiunti, sono considerati assigned_tags; conv_tags e auto_tags non possono essere manipolati.
        if chunk_id.startswith("predq_"):
            pred_queries = data.setdefault("predetermined_queries", {})
            query_entry = pred_queries.setdefault(chunk_id, {})
            current_tags = query_entry.setdefault("graphically_assigned_tags", [])
    
            updated = False
            for t in tags:
                if t not in current_tags:
                    current_tags.append(t)
                    updated = True
    
            if updated:
                self.save_data(data)
            return
    
        # Gestione Chunk Standard del Database
        if "global_tags" not in data:
            data["global_tags"] = {}
        if "tag_overrides" not in data:
            data["tag_overrides"] = {}
    
        if chunk_id not in data["tag_overrides"]:
            data["tag_overrides"][chunk_id] = {"user_tags": []}
    
        current_tags = data["tag_overrides"][chunk_id].setdefault("user_tags", [])
    
        updated = False
        for t in tags:
            if t not in current_tags:
                current_tags.append(t)
                updated = True
    
                if t not in data["global_tags"]:
                    used_colors = {
                        t_info.get("color") for t_info in data["global_tags"].values()
                        if isinstance(t_info, dict)
                    }
                    available = [c for c in self.DEFAULT_PALETTE if c not in used_colors]
                    col = (
                        available[0] if available
                        else self.DEFAULT_PALETTE[len(data["global_tags"]) % len(self.DEFAULT_PALETTE)]
                    )
                    data["global_tags"][t] = {"count": 1, "color": col}
                else:
                    data["global_tags"][t]["count"] += 1
    
        if updated:
            self.save_data(data)

    def add_tag_overrides_batch(self, chunk_tags_map: Dict[str, List[str]]) -> None:
        """
        Assegna tag a più chunk in un'unica operazione di lettura+scrittura su disco.
        chunk_tags_map: {chunk_id: [tag1, tag2, ...], ...}
        """
        if not chunk_tags_map:
            return
    
        data = self.load_data()
        if "tag_overrides" not in data:
            data["tag_overrides"] = {}
        if "global_tags" not in data:
            data["global_tags"] = {}
    
        updated = False
    
        for chunk_id, tags in chunk_tags_map.items():
            if not tags:
                continue
    
            cid_str = str(chunk_id)
    
            # Ramo Query Pre-Determinata
            if cid_str.startswith("predq_"):
                pred_queries = data.setdefault("predetermined_queries", {})
                query_entry = pred_queries.setdefault(cid_str, {})
                current_tags = query_entry.setdefault("graphically_assigned_tags", [])
    
                for t in tags:
                    if t not in current_tags:
                        current_tags.append(t)
                        updated = True
                continue
    
            # Ramo Chunk Standard
            if cid_str not in data["tag_overrides"]:
                data["tag_overrides"][cid_str] = {"user_tags": []}
    
            current_tags = data["tag_overrides"][cid_str].setdefault("user_tags", [])
    
            for t in tags:
                if t not in current_tags:
                    current_tags.append(t)
                    updated = True
    
                    if t not in data["global_tags"]:
                        used_colors = {
                            t_info.get("color") for t_info in data["global_tags"].values()
                            if isinstance(t_info, dict)
                        }
                        available = [c for c in self.DEFAULT_PALETTE if c not in used_colors]
                        col = (
                            available[0] if available
                            else self.DEFAULT_PALETTE[len(data["global_tags"]) % len(self.DEFAULT_PALETTE)]
                        )
                        data["global_tags"][t] = {"count": 1, "color": col}
                    else:
                        data["global_tags"][t]["count"] += 1
    
        if updated:
            self.save_data(data)

    def remove_tag_override(self, chunk_id: str, tag: str):
        """
        Rimuove un tag da un chunk o da una query pre-determinata, decrementando il contatore globale.
        Se questo scende a 0, rimuove il tag da tale registro globale.
        Decrementa global_tags SOLO per i chunk del database.
        """
        data = self.load_data()
        chunk_id = str(chunk_id)
    
        # Rimozione da Query Pre-Determinata
        if chunk_id.startswith("predq_"):
            pred_queries = data.get("predetermined_queries", {})
            if chunk_id in pred_queries:
                current_tags = pred_queries[chunk_id].get("graphically_assigned_tags", [])
                if tag in current_tags:
                    current_tags.remove(tag)
                    self.save_data(data)
            return
    
        # Rimozione da Chunk Standard
        tag_removed = False
        if "tag_overrides" in data and chunk_id in data["tag_overrides"]:
            current_tags = data["tag_overrides"][chunk_id].get("user_tags", [])
            if tag in current_tags:
                current_tags.remove(tag)
                tag_removed = True
                if not current_tags:
                    del data["tag_overrides"][chunk_id]
    
        if tag_removed and "global_tags" in data and tag in data["global_tags"]:
            data["global_tags"][tag]["count"] -= 1
            if data["global_tags"][tag]["count"] <= 0:
                del data["global_tags"][tag]
            self.save_data(data)

    ### RESET file sidecar
    def reset_all(self):
        """Ripristina il file sidecar azzerando le modifiche (UNDO globale)."""
        self.save_data({
            "modified_adjacencies": {},
            "tag_overrides": {},
            "global_tags": {},
            "predetermined_queries": {}
        })
        print("<<! File sidecar ripristinato allo stato iniziale !>>")

    ### Gestione Query Pre-Determinate
    # Estrazione dati
    def get_predetermined_queries(self) -> Dict[str, Dict[str, Any]]:
        """Restituisce l'intero dizionario delle query pre-determinate salvate."""
        data = self.load_data()
        return data.get("predetermined_queries", {})

    def get_predetermined_query(self, pred_query_id: str) -> Dict[str, Any]:
        """Restituisce i dati e le adiacenze per una specifica query pre-determinata."""
        queries = self.get_predetermined_queries() or {}
        return queries.get(str(pred_query_id), {})

    def get_predetermined_query_adjacencies(self, pred_query_id: str) -> Dict[str, Dict[str, str]]:
        """
        Restituisce il dizionario delle adiacenze modificate per una specifica query pre-determinata.
        Format di ritorno:
        {'chunk_12': {'distance_class': 'AVVICINATI'}}
        """
        query_entry = self.get_predetermined_query(pred_query_id)
        return query_entry.get("adjacencies", {})

    # Salvataggio adiacenze
    def save_predetermined_query_adjacency(self, chunk_id_1: str, chunk_id_2: str, distance_class: Optional[str]):
        """
        Salva o aggiorna l'adiacenza tra una query pre-determinata (ID con prefisso 'predq_') e un chunk.
        Se la classe e' INVARIATI, rimuove la voce dal JSON.
        """
        c1, c2 = str(chunk_id_1), str(chunk_id_2)
        if c1 == c2:
            return

        if c1.startswith("predq_"):
            pred_id, target_chunk = c1, c2
        elif c2.startswith("predq_"):
            pred_id, target_chunk = c2, c1
        else:
            self.save_pairwise_class_edits(c1, c2, distance_class)
            return

        data = self.load_data()
        pred_queries = data.setdefault("predetermined_queries", {})
        query_entry = pred_queries.setdefault(pred_id, {"adjacencies": {}, "graphically_assigned_tags": []})
        adj = query_entry.setdefault("adjacencies", {})

        norm_class = str(distance_class).upper().strip() if distance_class is not None else "INVARIATI"
        if norm_class not in DISTANCE_CLASS_FACTORS:
            norm_class = "INVARIATI"

        if norm_class == "INVARIATI":
            if target_chunk in adj:
                del adj[target_chunk]
            print(f"<<| Adiacenza query pre-determinata rimossa (INVARIATI) tra [{pred_id}] e [{target_chunk}] |>>")
        else:
            adj[target_chunk] = {"distance_class": norm_class}
            print(f"<<| Modifica adiacenza query pre-determinata salvata tra [{pred_id}] e [{target_chunk}]: classe={norm_class} |>>")

        self.save_data(data)

    def save_predetermined_query_adjacencies_batch(self, edits: List[Dict[str, Any]]):
        """Salva un blocco di modifiche di adiacenza per query pre-determinate in un'unica operazione di I/O."""
        if not edits or not isinstance(edits, list):
            return

        data = self.load_data()
        pred_queries = data.setdefault("predetermined_queries", {})
        updated = False

        for edit in edits:
            if not edit or "chunk_1" not in edit or "chunk_2" not in edit:
                continue
            c1, c2 = str(edit["chunk_1"]), str(edit["chunk_2"])
            if c1 == c2:
                continue

            if c1.startswith("predq_"):
                pred_id, target_chunk = c1, c2
            elif c2.startswith("predq_"):
                pred_id, target_chunk = c2, c1
            else:
                continue

            query_entry = pred_queries.setdefault(pred_id, {"adjacencies": {}, "graphically_assigned_tags": []})
            adj = query_entry.setdefault("adjacencies", {})

            d_cls = edit.get("distance_class")
            norm_class = str(d_cls).upper().strip() if d_cls is not None else "INVARIATI"
            if norm_class not in DISTANCE_CLASS_FACTORS:
                norm_class = "INVARIATI"

            if norm_class == "INVARIATI":
                if target_chunk in adj:
                    del adj[target_chunk]
            else:
                adj[target_chunk] = {"distance_class": norm_class}
            updated = True

        if updated:
            self.save_data(data)
            print(f"<<| Batch adiacenze query pre-determinate salvato: {len(edits)} elementi processati |>>")
EOF

In [2]:
%%bash
cat << 'EOF' > ~/tesi_graphrag/src/predetermined_query_handling_methods.py
import sys
import json
import logging
from pathlib import Path
from typing import Optional, Tuple, Dict, Any, Union, List
import numpy as np

project_root = Path("~/tesi_graphrag").expanduser().resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.config import PREDET_QUERY_THRESHOLD
from src.sidecar_manager import SidecarManager

logger = logging.getLogger(__name__)

def compute_cosine_similarity(vec1: Union[List[float], np.ndarray], vec2: Union[List[float], np.ndarray]) -> float:
    """Calcola la similarità coseno tra due vettori."""
    v1 = np.array(vec1, dtype=float)
    v2 = np.array(vec2, dtype=float)
    norm1 = np.linalg.norm(v1)
    norm2 = np.linalg.norm(v2)
    if norm1 == 0.0 or norm2 == 0.0:
        return 0.0
    return float(np.dot(v1, v2) / (norm1 * norm2))

def verify_if_pred_query(
    query_text: str,
    sidecar_manager: SidecarManager,
    query_vector: Optional[Union[List[float], np.ndarray]] = None,
    embedder: Optional[Any] = None,
    threshold: float = PREDET_QUERY_THRESHOLD
) -> Tuple[Optional[str], Optional[Dict[str, Any]]]:
    """
    Verifica se una query utente corrisponde a una query pre-determinata salvata nel Sidecar.
    
    Returns:
        Tuple[Optional[str], Optional[Dict[str, Any]]]:
            Una tupla (predq_id, query_data) dove:
            - predq_id (str || None): L'ID univoco della query pre-determinata riconosciuta (es. 'predq_101'),
              oppure None se non viene trovata alcuna corrispondenza.
            - query_data (Dict[str, Any] || None): Il dizionario completo estratto dal file Sidecar
              contenente i metadati e le modifiche legate alla query, oppure None se non trovata.

            In caso di match (predq_id != None), 'query_data' include le seguenti chiavi:
                > "query_text" (str): Il testo originale della query memorizzata.
                > "adjacencies" (Dict[str, Dict[str, str]]): Le adiacenze salvate tra il nodo query
                  e i chunk, con le relative classi di distanza modificate 
                  (es. {"chunk_12": {"distance_class": "AVVICINATI"}}).
                > "graphically_assigned_tags" (List[str]): Lista dei tag grafici/manuali 
                  assegnati dall'utente a questo specifico nodo query.
                > "vector" / "embedding" (List[float] | None): Il vettore di embedding 
                  della query memorizzato nel sidecar.
    """
    if not query_text or not isinstance(query_text, str):
        return None, None

    pred_queries = sidecar_manager.get_predetermined_queries()
    if not pred_queries:
        return None, None

    norm_input_text = query_text.strip().lower()

    # Tentativo match testuale esatto
    for pred_id, data in pred_queries.items():
        stored_text = data.get("query_text", "") or data.get("text", "")
        if stored_text and stored_text.strip().lower() == norm_input_text:
            logger.info(f"<<| Match testuale esatto per Query Pre-Determinata: [{pred_id}] |>>")
            return pred_id, data

    # Tentativo con cos_sim
    if query_vector is None and embedder is not None:
        try:
            if hasattr(embedder, "embed_query"):
                query_vector = embedder.embed_query(query_text)
            elif hasattr(embedder, "encode"):
                query_vector = embedder.encode(query_text)
            elif callable(embedder):
                query_vector = embedder(query_text)
        except Exception as e:
            logger.warning(f"<[WARNING]> Errore nella generazione dell'embedding di query: {e}")
            query_vector = None

    if query_vector is not None:
        best_pred_id = None
        best_data = None
        max_sim = -1.0

        for pred_id, data in pred_queries.items():
            stored_vector = data.get("vector") or data.get("embedding")
            if stored_vector is not None:
                sim = compute_cosine_similarity(query_vector, stored_vector)
                if sim > max_sim:
                    max_sim = sim
                    best_pred_id = pred_id
                    best_data = data

        if max_sim >= threshold and best_pred_id is not None:
            logger.info(f"<<| Match vettoriale trovato per Query Pre-Determinata [{best_pred_id}] con similarità {max_sim:.4f} (>= {threshold:.2f}) |>>")
            return best_pred_id, best_data

    return None, None

def add_predetermined_query(
    query_text: str,
    sidecar_manager: SidecarManager,
    query_id: Optional[str] = None,
    tags: Optional[List[str]] = None,
    embedder: Optional[Any] = None
) -> str:
    """
    Aggiunge una query pre-determinata salvando testo ed embedding nel Sidecar.
    """
    if not query_text or not isinstance(query_text, str):
        raise ValueError("!!!> query_text non valido o vuoto.")

    norm_input_text = query_text.strip().lower()
    data = sidecar_manager.load_data()
    pred_queries = data.setdefault("predetermined_queries", {})

    # Controllo duplicati
    for existing_id, existing_data in pred_queries.items():
        stored_text = existing_data.get("query_text", "") or existing_data.get("text", "")
        if stored_text and stored_text.strip().lower() == norm_input_text:
            logger.info(f"i>> Query già presente nel Sidecar con ID '[{existing_id}]': '{query_text}'")
            return existing_id

    # Per garantire prefisso 'predq_'
    if query_id:
        q_id_str = str(query_id).strip()
        final_id = q_id_str if q_id_str.startswith("predq_") else f"predq_{q_id_str}"
    else:
        existing_indices = [
            int(k.replace("predq_", ""))
            for k in pred_queries.keys()
            if k.startswith("predq_") and k.replace("predq_", "").isdigit()
        ]
        next_idx = max(existing_indices, default=0) + 1
        final_id = f"predq_{next_idx}"

    if final_id in pred_queries:
        logger.warning(f"!>> L'ID '[{final_id}]' è già presente nel Sidecar.")
        return final_id

    # Calcolo embedding
    vector = None
    if embedder is not None:
        try:
            if hasattr(embedder, "embed_query"):
                vector = embedder.embed_query(query_text)
            elif hasattr(embedder, "encode"):
                vector = embedder.encode(query_text)
            elif callable(embedder):
                vector = embedder(query_text)
        except Exception as e:
            logger.warning(f"!!!> Impossibile generare l'embedding per '{query_text}': {e}")

    if vector is not None and hasattr(vector, "tolist"):
        vector = vector.tolist()

    # Inizializzazione struttura 
    pred_queries[final_id] = {
        "query_text": query_text,
        "vector": vector,
        "graphically_assigned_tags": [],
        "adjacencies": {}
    }
    sidecar_manager.save_data(data)

    # Assegnazione tag
    if tags:
        sidecar_manager.add_tag_override_for_chunk(final_id, tags)

    logger.info(f"<<| Query Pre-Determinata registrata: ID=[{final_id}] |>>")
    return final_id


### 3. Conversione Batch da File (es. eval_queries.json)

def add_predetermined_queries_from_file(
    queries_path: Union[str, Path],
    sidecar_manager: SidecarManager,
    embedder: Optional[Any] = None
) -> List[str]:
    """
    Legge un file JSON di query (es. eval_queries.json) e le converte tutte in query pre-determinate.
    """
    path = Path(queries_path).expanduser().resolve()
    if not path.exists():
        logger.error(f"!!!> File di query non trovato: {path}")
        return []

    with open(path, "r", encoding="utf-8") as f:
        eval_queries = json.load(f)

    if not isinstance(eval_queries, list):
        logger.error("!!!> Formato JSON non valido: ci si attende una lista.")
        return []

    added_ids = []
    for q_entry in eval_queries:
        if not isinstance(q_entry, dict):
            continue

        query_text = q_entry.get("query") or q_entry.get("query_text") or q_entry.get("text")
        query_id = q_entry.get("id") or q_entry.get("query_id")
        
        expected_tag = q_entry.get("expected_tag")
        tags = [expected_tag] if expected_tag else []

        if query_text:
            pred_id = add_predetermined_query(
                query_text=query_text,
                sidecar_manager=sidecar_manager,
                query_id=query_id,
                tags=tags,
                embedder=embedder
            )
            added_ids.append(pred_id)

    logger.info(f"<<| Convertite {len(added_ids)} query da '{path.name}' |>>")
    return added_ids
    
EOF

In [3]:
%%bash
cat << 'EOF' > ~/tesi_graphrag/src/distance_tag_reranker.py
import sys
import logging
from pathlib import Path
from typing import Any, Dict, List, Optional, Union
import numpy as np

from qdrant_client import QdrantClient

project_root = Path("~/tesi_graphrag").expanduser().resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.sidecar_manager import SidecarManager
from src.distance_retriever import DistanceRetriever, build_query_tags
from src.pred_query_verifier import verify_if_pred_query
from src.config import (
    COLLECTION_NAME,
    TAG_BOOST_ASSIGNED_CONV, TAG_BOOST_AUTO, TAG_MAX_BOOST,
    DISTANCE_CLASS_FACTORS
    RETRIEVAL_TOP_K, RERANKING_TOP_N,
)

logger = logging.getLogger(__name__)

def _extract_chunk_id(hit: Any, fallback_index: Optional[int] = None) -> str:
    """Estrae l'ID del chunk da un oggetto Qdrant (ScoredPoint o Record)."""
    payload = getattr(hit, "payload", {}) or {}
    meta = payload.get("metadata") if isinstance(payload.get("metadata"), dict) else payload

    chunk_id = payload.get("chunk_id") or meta.get("chunk_id")
    if not chunk_id:
        doc_id = payload.get("doc_id") or meta.get("doc_id") or "doc"
        chunk_idx = payload.get("chunk_index") if payload.get("chunk_index") is not None else meta.get("chunk_index")
        if chunk_idx is None:
            chunk_idx = fallback_index
        chunk_id = f"{doc_id}_chunk_{chunk_idx}"

    return chunk_id

class DistanceTagReranker:
    """
    Reranker basato sui Tag in comune, che inoltre fa uso delle distanze modificate (Classi di Distanza) per recuperare
    ulteriori chunk qualora quelli originari non fossero sufficienti.
    I chunk originariamente recuperati e quelli recuperati tramite il distance_retriever vengono poi suddivisi in 4 categorie,
    la cui importanza è 1 > 2 > 3 > 4:
            - categoria 1: i chunk del 1° retrieval che posseggono i conversation_tags (nel loro intero se più di uno)
            - categoria 2: i chunk del 2° retrieval che posseggono i conversation_tags (nel loro intero se più di uno)
            - categoria 3: i chunk del 2° retrieval che NON condividono i conversation_tags; un possesso parziale è usato per assegnare bonus
            - categoria 4: i chunk del 1° retrieval che NON condividono i conversation_tags; un possesso parziale è usato per assegnare bonus

    Combina i candidati del 1° retrieval (vettoriale puro) e del 2° retrieval (distance_retriever),
    riordinandoli internamente per ciascuna categoria e troncando la lista finale ai TOP_J richiesti.

    Ciascuna categoria è ordinata per final_score, il quale è calcolato secondo i criteri passati da ORDER_BY; supponendo "TAGS" "DIST:
            - categoria 1: ordinata secondo la sola sovrapposizione di assigned e automatic tags, con pesi relativi; la presenza dei conv_tag è ovviamente scontata
            - categoria 2: ordinata secondo la sovrapposizione di assigned e automatic tags, più il distance_factor; presenza dei conv_tag scontata
            - categoria 3: ordinata secondo la sovrapposizione di conversation tags parziali e assigned e automatic tags, più il distance_factor
            - categoria 4: ordinata secondo la sovrapposizione di conversation tags parziali e assigned e automatic tags  
    """

    def __init__(
        self,
        qdrant_client: QdrantClient,
        sidecar_manager: SidecarManager,
        collection_name: str=COLLECTION_NAME,
        distance_retriever: Optional[DistanceRetriever] = None,
        embeddings: Optional[Any] = None,
    ):
        if sidecar_manager is None:
            raise ValueError("!!!>>> sidecar_manager è obbligatorio e non può essere None.")
        if qdrant_client is None:
            raise ValueError("!!!>>> qdrant_client è obbligatorio e non può essere None.")

        self.qdrant_client = qdrant_client
        self.collection_name = collection_name
        self.sidecar_manager = sidecar_manager
        self.embedder = embeddings
        self.distance_retriever = distance_retriever or (
            DistanceRetriever(
                qdrant_client=qdrant_client,
                collection_name=collection_name,
                sidecar_manager=self.sidecar_manager,
            )
        )

    def generic_rerank(
        self,
        retrieved_points: List[Any],
        order_by: List[str],
        top_j: int,
        query_vector: Union[List[float], np.ndarray],
        query_tags: Dict[str, List[str]],
        distance_retrieved_records: Optional[List[Dict[str, Any]]] = None,
        top_m: Optional[int]= None,
        query_text: Optional[str] = None,
        embedder: Optional[Any] = None
    ) -> Dict[str, List[Dict[str, Any]]]:
        """
        Esegue il reranking dividendo in 4 categorie, distintamente rioridnate secondo i criteri di ORDER_BY e restituisce
        i TOP_J post-reorder.
        Se stiamo analizzando una query pre-determinata, allora si aggiunge al final score il fattore moltiplicativo
        della DistanceClass del chunk in analisi dalla query - vale per tutte e 4 le categorie, indipendentemente dai criteri di order_by
        Tra di questi, se c'è posto, ci sono i TOP_M recuperati tramite il distance_retriever.

        Args:
            retrieved_points: Punti recuperati da Qdrant tramite retrieval vettoriale.
                              Devono essere oggetti di Qdrant; configurato per gestire quanto di seguito:
                              response = qdrant_client.query_points(
                                  collection_name=COLLECTION_NAME,
                                  query=query_vector,
                                  limit=PERSONALISED_TOP_K,
                                  with_vectors=True,
                                  with_payload=True
                              )
                              raw_records = response.points
                              -> .generic_rerank(retrieved_points=raw_records, ...)
            distace_retrieved_records: Record recuperati tramite distance_retrieval; se non passati, vengono calcolati.
                                       Devono essere dizionari di return di distance_retriever.retrieve_close()
            order_by: Lista di flag che regolano filtri e moltiplicatori:
                    - "CONV_TAG_ONLY" (esclude nativamente categorie 3 e 4): Filtro rigido (ALL match sui conversation_tags).
                    - "DIST" (per le categorie 2 e 3): Applica il fattore moltiplicativo della classe di distanza.
                    - "TAGS" (per le categorie 1-4): Applica bonus di sovrapposizione dei tag.
                    - "SCORE" o "" (per le categorie 1-4): Calcola unicamente la similarità coseno base. Se presente con altri, questo NON ha la priorità.
            top_j: Numero massimo di vicini da restituire dopo l'ordinamento.
            query_vector: Vettore di embedding della query.
            query_tags: Dizionario contenente i tag della query divisi per categoria:
                        {"conversation_tags": [...], "assigned_tags": [...],
                         "automatic_tags": [...], "all_tags": [...]}.
            top_m: Eventuale valore di override di quello calcolato nel metodo che impone un numero massimo di vicini da recuperare e restituire.
            query_text: Se passato si abilita il confronto del testo con le query pre-determinate e la loro gestione nel reranking
            embedder: Il modello di embedding da utilizzare per il confronto di query_text con quelle pre-determinate.
            
        Returns:
            Dizionario con
                    -[Debug] Lista di dizionari descrittivi con final_score calcolato e ordinati per categorie e in modo decrescente.
                    -I TOP_J richiesti di tale lista
                    -[Debug] I punti recuperati dal distance_retriever
        """
        empty_response = {
            "top_j_ranked": [],
            "final_ranked": [],
            "distance_retrieved_records": []
        }
        if not retrieved_points and not distance_retrieved_records:
            return empty_response

        if top_j is None or top_j <= 0:
            logger.warning("!>> Reranking annullato per il valore di TOP_J non valido: %s", top_j)
            return empty_response

        # Controllo predq-reranker
        predq = False
        predq_id, predq_data = None, None
        if query_text:
            embeddings = embedder or self.embedder
            predq_id, predq_data = verify_if_pred_query(
                query_text=query_text,
                sidecar_manager=self.sidecar_manager,
                query_vector=query_vector,
                embedder=embeddings
            )
            if predq_id:
                predq = True
            

        # Rimozione eventuali tag duplicati nelle gerarchie inferiori tramite build_query_tags
        query_tags = query_tags or {} # fallback per sicurezza a dict vuoto
        if predq and predq_data:
            # Aggiornamento query_tags con i tag assegnati graficamente alla pred_query
            pred_ass_tags = pred_data.get("graphically_assigned_tags") or []
            combined_ass_tags = list(set(query_tags.get("assigned_tags", []) + pred_ass_tags))
    
            query_tags = build_query_tags(
                conversation_tags=query_tags.get("conversation_tags", []),
                assigned_tags=combined_ass_tags,
                automatic_tags=query_tags.get("automatic_tags", [])
            )
        else:
            query_tags = build_query_tags(
                conversation_tags=query_tags.get("conversation_tags", []),
                assigned_tags=query_tags.get("assigned_tags", []),
                automatic_tags=query_tags.get("automatic_tags", []),
            )

        # Normalizzazione flag di order_by
        order_by_to_upper = [str(item).upper().strip() for item in (order_by or []) if item]

        sidecar_data = self.sidecar_manager.load_data()
        tag_overrides = sidecar_data.get("tag_overrides", {})

        conv_tags = query_tags.get("conversation_tags", [])
        ass_tags = query_tags.get("assigned_tags", [])
        auto_tags = query_tags.get("automatic_tags", [])
        all_query_tags = query_tags.get("all_tags", [])

        conv_set = set(conv_tags)
        ass_set = set(ass_tags)
        auto_set = set(auto_tags)
        all_query_set = set(all_query_tags)

        ### Partizionamento in categorie 1 e 4 dei retrieved_points
        category1_candidates = []
        category4_candidates = []
        for idx, hit  in enumerate(retrieved_points or []):
            payload = getattr(hit, "payload", {}) or {}
            chunk_id = _extract_chunk_id(hit, fallback_index=idx)

            # cos_sim con query
            initial_score = max(0.0, float(getattr(hit, "score", 0.0)))

            chunk_info = tag_overrides.get(chunk_id, {})
            chunk_tags = chunk_info.get("user_tags", payload.get("user_tags", []))
            chunk_tags_set = set(chunk_tags)

            # Requisito conversation_tag
            if "CONV_TAG_ONLY" in order_by_to_upper and conv_tags:
                if not conv_set.issubset(chunk_tags_set):
                    continue

            # Controllo generale appartenenza a categoria 1 o 4 (se chunk ha il conv_tag o meno)
            has_all_conv_tag = bool(conv_tags and conv_set.issubset(chunk_tags_set))

            # Recupero vettore da inserire in output
            raw_vector = getattr(hit, "vector", None)
            if isinstance(raw_vector, dict): # gestisce sia liste/array che dict come vettori
              chunk_vector = next(iter(raw_vector.values())) if raw_vector else None
            else:
              chunk_vector = raw_vector

            # Distance Factor inutile in questa fase
            dist_factor = 1.0

            # Moltiplicatore per TAG coincidenti
            tag_alteration = 1.0
            matched_all = []
            if "TAGS" in order_by_to_upper:
                matched_conv = list(conv_set.intersection(chunk_tags_set))
                matched_ass = list(ass_set.intersection(chunk_tags_set))
                matched_auto = list(auto_set.intersection(chunk_tags_set))
                matched_all = list(all_query_set.intersection(chunk_tags_set))

                bonus_conv = len(matched_conv) * TAG_BOOST_ASSIGNED_CONV
                bonus_ass = len(matched_ass) * TAG_BOOST_ASSIGNED_CONV
                bonus_auto = len(matched_auto) * TAG_BOOST_AUTO

                if not has_all_conv_tag: # categoria 4: contano i match parziali
                    total_bonus = bonus_conv + bonus_ass + bonus_auto
                else: # categoria 1: tutti fanno perfect_match con il conv_tag, contano solo assigned e auto
                    total_bonus = bonus_ass + bonus_auto

                if total_bonus > 0:
                    tag_alteration = 1.0 + min(total_bonus, TAG_MAX_BOOST)

            # Moltiplicatore VICINANZA a query pre-determinata
            predq_alteration = 1.0
            if predq:
                predq_adj = predq_data.get("adjacencies") or {}
                if chunk_id in predq_adj:
                    d_class = predq_adj[chunk_id].get("distance_class", "INVARIATI")
                    predq_alteration = DISTANCE_CLASS_FACTORS.get(d_class, 1.0)
            
            # Punteggio finale
            raw_final_score = initial_score * dist_factor * tag_alteration * predq_alteration

            candidate_item = {
                "chunk_id": chunk_id,
                "point": hit,
                "initial_score": round(initial_score, 4),
                "dist_factor": round(dist_factor, 3),
                "tag_alteration": round(tag_alteration, 3),
                "predq_alteration": round(predq_alteration, 3),
                "final_score": round(raw_final_score, 4),
                "_raw_final_score": raw_final_score,
                "source_retrieval": "vectorial_retriever",
                "category": 1 if has_all_conv_tag else 4,
                "vector": chunk_vector,
                "payload": payload,
                "chunk_tags": chunk_tags
            }

            if has_all_conv_tag:
                category1_candidates.append(candidate_item)
            else:
                category4_candidates.append(candidate_item)

        ### Distance Retrieval se necessario per partizionamento in categorie 2 e 3
        if distance_retrieved_records is None:
            n_category1 = len(category1_candidates)
            if top_m is not None:
                effective_top_m = top_m # Override
            else:
                effective_top_m = max(0, top_j - n_category1)

            # Necessità il distance_retrieval
            if effective_top_m > 0 and category1_candidates:
                to_analyze_nodes = category1_candidates
                if to_analyze_nodes and query_vector is not None:
                    distance_retrieved_records = self.distance_retriever.retrieve_close(
                        to_analyze=to_analyze_nodes,
                        order_by=order_by_to_upper,
                        top_m=effective_top_m,
                        query_vector=query_vector,
                        query_tags=query_tags,
                        predq_id=predq_id if predq else None
                    )

        ### Sorting dei candidati per categorie 1 e 4 (2 e 3 già ordinate da retrieve_close())
        category1_candidates.sort(key=lambda x: x["_raw_final_score"], reverse=True)
        category4_candidates.sort(key=lambda x: x["_raw_final_score"], reverse=True)

        # Deduplicazione di eventuali chunk della cat4 presenti anche in cat3.
        dist_retrieved_ids = {item["chunk_id"] for item in (distance_retrieved_records or [])}
        category4_candidates = [c for c in category4_candidates if c["chunk_id"] not in dist_retrieved_ids]

        final_ranked = category1_candidates + (distance_retrieved_records or []) + category4_candidates

        for item in final_ranked:
            if "_raw_final_score" in item:
                del item["_raw_final_score"]

        ### Return per debug e wrapping
        return {
            "top_j_ranked": final_ranked[:top_j],
            "final_ranked": final_ranked,
            "distance_retrieved_records": distance_retrieved_records or [],
        }
    
    def rerank_top_j(
        self,
        retrieved_points: List[Any],
        order_by: List[str],
        top_j: int,
        query_vector: Union[List[float], np.ndarray],
        query_tags: Dict[str, List[str]],
        distance_retrieved_records: Optional[List[Dict[str, Any]]] = None,
        top_m: Optional[int]= None,
        query_text: Optional[str] = None,
        embedder: Optional[Any] = None
    ) -> List[Dict[str, Any]]:
        """
        Wrapper di generic_rerank() che returna di default solo i TOP_J records
        """
        rrnk_rslt = self.generic_rerank(
            retrieved_points=retrieved_points,
            order_by=order_by,
            top_j=top_j,
            query_vector=query_vector,
            query_tags=query_tags,
            distance_retrieved_records=distance_retrieved_records,
            top_m=top_m,
            query_text=query_text,
            embedder=embedder
        )
        return rrnk_rslt.get("top_j_ranked", [])

    def rerank_top_k(
        self,
        retrieved_points: List[Any],
        order_by: List[str],
        query_vector: Union[List[float], np.ndarray],
        query_tags: Dict[str, List[str]],
        top_k: int=RETRIEVAL_TOP_K,
        distance_retrieved_records: Optional[List[Dict[str, Any]]] = None,
        top_m: Optional[int]= None,
        query_text: Optional[str] = None,
        embedder: Optional[Any] = None
    ) -> List[Dict[str, Any]]:
        """
        Wrapper di generic_rerank() che returna di default solo src.config.RETRIEVAL_TOP_K records
        """
        return self.rerank_top_j(
            retrieved_points=retrieved_points,
            order_by=order_by,
            top_j=top_k,
            query_vector=query_vector,
            query_tags=query_tags,
            distance_retrieved_records=distance_retrieved_records,
            top_m=top_m,
            query_text=query_text,
            embedder=embedder
        )

    def rerank(
        self,
        retrieved_points: List[Any],
        order_by: List[str],
        query_vector: Union[List[float], np.ndarray],
        query_tags: Dict[str, List[str]],
        top_n: int=RERANKING_TOP_N,
        distance_retrieved_records: Optional[List[Dict[str, Any]]] = None,
        top_m: Optional[int]= None,
        query_text: Optional[str] = None,
        embedder: Optional[Any] = None
    ) -> List[Dict[str, Any]]:
        """
        Wrapper di generic_rerank() che returna di default solo src.config.RERANKING_TOP_N records
        """
        return self.rerank_top_j(
            retrieved_points=retrieved_points,
            order_by=order_by,
            top_j=top_n,
            query_vector=query_vector,
            query_tags=query_tags,
            distance_retrieved_records=distance_retrieved_records,
            top_m=top_m,
            query_text=query_text,
            embedder=embedder
        )
EOF

In [4]:
%%bash
cat << 'EOF' > ~/tesi_graphrag/src/distance_retriever.py
import sys
import uuid
import logging
from pathlib import Path
from typing import Any, Dict, List, Optional, Union
import numpy as np

from qdrant_client import QdrantClient

project_root = Path("~/tesi_graphrag").expanduser().resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
from src.sidecar_manager import SidecarManager
from src.config import (
    COLLECTION_NAME,
    DISTANCE_CLASS_FACTORS,
    TAG_BOOST_ASSIGNED_CONV, TAG_BOOST_AUTO, TAG_MAX_BOOST,
    MIN_COSINE_THRESHOLD
)

logger = logging.getLogger(__name__)

def cosine_sim(v1: Union[List[float], np.ndarray], v2: Union[List[float], np.ndarray]) -> float:
    """Calcola la Cosine Similarity tra due vettori."""
    arr1 = np.array(v1, dtype=float)
    arr2 = np.array(v2, dtype=float)
    norm1 = np.linalg.norm(arr1)
    norm2 = np.linalg.norm(arr2)
    if norm1 == 0.0 or norm2 == 0.0:
        return 0.0
    return float(np.dot(arr1, arr2) / (norm1 * norm2))

# Per garantire che i query_tag non si ripetano nelle varie categorie
def build_query_tags(
    conversation_tags: Optional[List[str]] = None,
    assigned_tags: Optional[List[str]] = None,
    automatic_tags: Optional[List[str]] = None,
) -> Dict[str, List[str]]:
    """
    Costruisce il dizionario query_tags garantendo la disgiunzione gerarchica dei tag:
    conversation_tags > assigned_tags > automatic_tags.
    """
    seen = set()

    clean_conv: List[str] = []
    for t in conversation_tags or []:
        if t and t not in seen:
            clean_conv.append(t)
            seen.add(t)

    clean_assigned: List[str] = []
    for t in assigned_tags or []:
        if t and t not in seen:
            clean_assigned.append(t)
            seen.add(t)

    clean_auto: List[str] = []
    for t in automatic_tags or []:
        if t and t not in seen:
            clean_auto.append(t)
            seen.add(t)

    all_tags: List[str] = clean_conv + clean_assigned + clean_auto

    return {
        "conversation_tags": clean_conv,
        "assigned_tags": clean_assigned,
        "automatic_tags": clean_auto,
        "all_tags": all_tags,
    }

class DistanceRetriever:
    """
    Modulo di 2° Retrieval basato sulle sole adiacenze e distanze modificate nel sidecar.
    Recupera da Qdrant i chunk vicini ("AVVICINATI" / "MOLTO_AVVICINATI") rispetto
    ai nodi estratti nel 1° retrieval, calcolandone e restituendo la cos_sim con la query.
    """

    def __init__(
        self,
        qdrant_client: QdrantClient,
        sidecar_manager: SidecarManager,
        collection_name: str = COLLECTION_NAME
    ):
        if sidecar_manager is None:
            raise ValueError("!!!>>> sidecar_manager è obbligatorio e non può essere None.")
        if qdrant_client is None:
            raise ValueError("!!!>>> qdrant_client è obbligatorio e non può essere None.")

        self.qdrant_client = qdrant_client
        self.sidecar_manager = sidecar_manager
        self.collection_name = collection_name

    def retrieve_close(
        self,
        to_analyze: List[Any],
        order_by: List[str],
        top_m: int,
        query_vector: Union[List[float], np.ndarray],
        query_tags: Dict[str, List[str]],
        predq_id: Optional[str] = None
    ) -> List[Dict[str, Any]]:
        """Esegue il retrieval su base di distanza e calcola il final_score per tutti i candidati adiacenti.

        Args:
            to_analyze: Punti da analizzare per trovare i vicini. Possono essere oggetti di Qdrant o dizionari
            order_by: Lista di flag che regolano filtri e moltiplicatori:
                    - "CONV_TAG_ONLY": Filtro rigido (ALL match sui conversation_tags).
                    - "DIST": Applica il fattore moltiplicativo della classe di distanza.
                    - "TAGS": Applica bonus di sovrapposizione dei tag.
                    - "SCORE" o "": Calcola unicamente la similarità coseno base. Se presente con altri, questo NON ha la priorità.
            top_m: Numero massimo di vicini da restituire dopo l'ordinamento.
            query_vector: Vettore di embedding della query.
            query_tags: Dizionario contenente i tag della query divisi per categoria:
                        {"conversation_tags": [...], "assigned_tags": [...],
                         "automatic_tags": [...], "all_tags": [...]}.
                        Se presente il predq_id, si presuppone che i graphically_assigned_tags siano già stati aggiunti.
            predq_id: l'ID della query pre-determinata che sta venendo eseguita (o la cui query posta era simile).
                      Se presente, vuol dire che stiamo analizzando una query pre-determinata. Se assente, la query è una qualunque.
                      Si considera nel calcolo del final_anche il DistanceFactor chunk-query, indipendentemente dal valore di order_by
                      La suddivisione in categorie non cambia, viene solo cambiato il punteggio interno.
            
        Returns:
            Lista di dizionari descrittivi con final_score calcolato e ordinati per categorie e in modo decrescente.
        """
        if (not to_analyze and not predq_id) or query_vector is None:
            return []

        if top_m is None or top_m <= 0:
            logger.warning("!>> Distance Retrieval annullato per il valore di TOP_M non valido: %s", top_m)
            return []

        # Rimozione eventuali tag duplicati nelle gerarchie inferiori -- 
        query_tags = query_tags or {} # se dovesse essere vuoto
        query_tags = build_query_tags(
            conversation_tags=query_tags.get("conversation_tags", []),
            assigned_tags=query_tags.get("assigned_tags", []),
            automatic_tags=query_tags.get("automatic_tags", []),
        )

        # Normalizzazione flag di order_by
        order_by_to_upper = [str(item).upper().strip() for item in (order_by or []) if item]

        sidecar_data = self.sidecar_manager.load_data()
        modified_adjacencies = sidecar_data.get("modified_adjacencies", {})
        tag_overrides = sidecar_data.get("tag_overrides", {})
        pred_queries = sidecar_data.get("predetermined_queries", {})

        # ID dei chunk da analizzare per evitare duplicazioni
        seen_ids = set()
        for idx, hit in enumerate(to_analyze):
            if isinstance(hit, dict): # Se hit è un dizionario
                payload = hit.get("payload", {}) or {}
                meta = (
                    payload.get("metadata")
                    if isinstance(payload.get("metadata"), dict)
                    else payload
                )
                chunk_id = (
                    hit.get("chunk_id") or payload.get("chunk_id") or meta.get("chunk_id")
                )
            else: # Se hit è un oggetto di Qdrant
                payload = getattr(hit, "payload", {}) or {}
                meta = (
                    payload.get("metadata")
                    if isinstance(payload.get("metadata"), dict)
                    else payload
                )
                chunk_id = payload.get("chunk_id") or meta.get("chunk_id")

            # Fallback se l'id ancora non è definito
            if not chunk_id:
                doc_id = payload.get("doc_id") or meta.get("doc_id") or "doc"
                chunk_idx = (
                    payload.get("chunk_index")
                    if payload.get("chunk_index") is not None
                    else meta.get("chunk_index")
                )
                chunk_id = f"{doc_id}_chunk_{chunk_idx}"

            seen_ids.add(chunk_id)

        # Mappatura adiacenze
        candidate_neighbors: Dict[str, Dict[str, Any]] = {}

        # Vicini a to_analyze
        for cid in seen_ids:
            if cid in modified_adjacencies:
                neighbors = modified_adjacencies[cid]
                for neighbor_id, info in neighbors.items():
                    if neighbor_id not in seen_ids:
                        d_class = info.get("distance_class", "INVARIATI")
                        if d_class in ["AVVICINATI", "MOLTO_AVVICINATI"]:
                            factor = DISTANCE_CLASS_FACTORS.get(d_class, 1.0)
                            # Prendiamo dist_factor massimo trai possibili
                            if neighbor_id not in candidate_neighbors or factor > candidate_neighbors[neighbor_id]["distance_factor"]:
                                candidate_neighbors[neighbor_id] = {
                                    "distance_class": d_class,
                                    "distance_factor": factor,
                                    "predq_alteration": 1.0
                                }

        # Vicini alla pred-query (se presente)
        if predq_id and predq_id in pred_queries:
            predq_data = pred_queries[predq_id] or {}
            predq_adj = predq_data.get("adjacencies") or {}
            for neighbor_id, info in predq_adj.items():
                if neighbor_id not in seen_ids:
                    d_class = info.get("distance_class", "INVARIATI")
                    if d_class in ["AVVICINATI", "MOLTO_AVVICINATI"]:
                        factor = DISTANCE_CLASS_FACTORS.get(d_class, 1.0)
                        if neighbor_id not in candidate_neighbors:
                            candidate_neighbors[neighbor_id] = {
                                "distance_class": d_class,
                                "distance_factor": 1.0,
                                "predq_alteration": factor
                            }
                        else:
                            existing_predq_factor = candidate_neighbors[neighbor_id].get("predq_alteration", 1.0)
                            if factor > existing_predq_factor:
                                candidate_neighbors[neighbor_id]["predq_alteration"] = factor
                        
        
        if not candidate_neighbors:
            return []

        # Recupero chunk da Qdrant via uuid
        chunk_id_to_uuid = {
            cid: str(uuid.uuid5(uuid.NAMESPACE_DNS, cid))
            for cid in candidate_neighbors.keys()
        }
        target_uuids = list(chunk_id_to_uuid.values())
        uuid_to_chunk_id = {v: k for k, v in chunk_id_to_uuid.items()}  # Mappa inversa

        try:
            retrieved_records = self.qdrant_client.retrieve(
                collection_name=self.collection_name,
                ids=target_uuids,
                with_payload=True,
                with_vectors=True,
            )
        except Exception as e:
            logger.error("!!!> [DistanceRetriever] Errore critico nel retrieval da Qdrant: %s", e, exc_info=True)
            return []

        # Liste tag
        conv_tags = query_tags.get("conversation_tags", [])
        assigned_tags = query_tags.get("assigned_tags", [])
        auto_tags = query_tags.get("automatic_tags", [])
        all_query_tags = query_tags.get("all_tags", [])

        # Elaborazione, filtraggio e calcolo dei final score,
        #  sui criteri di order_by
        category2_candidates = []
        category3_candidates = []
        for rec in retrieved_records:
            payload = getattr(rec, "payload", {}) or {}

            # Recupero chunk_id con mappa inversa
            rec_id_str = str(rec.id)
            chunk_id = uuid_to_chunk_id.get(rec_id_str)

            # Fallback calcolo chunk_id se non trovato
            if not chunk_id:
                meta = payload.get("metadata") if isinstance(payload.get("metadata"), dict) else payload
                doc_id = payload.get("doc_id") or meta.get("doc_id") or "doc"
                chunk_idx = payload.get("chunk_index") if payload.get("chunk_index") is not None else meta.get("chunk_index")
                chunk_id = payload.get("chunk_id") or meta.get("chunk_id") or f"{doc_id}_chunk_{chunk_idx}"

            # Recupero info chunk
            chunk_info = tag_overrides.get(chunk_id, {})
            chunk_tags = chunk_info.get("user_tags", payload.get("user_tags", []))

            # Requisito conversation_tag
            if "CONV_TAG_ONLY" in order_by_to_upper:
                # Richiesta inclusione totale dei conv_tags
                if conv_tags:
                    has_all = all(cnvtag in chunk_tags for cnvtag in conv_tags)
                    if not has_all:
                        continue

            # Controllo generale appartenenza a categoria 2 o 3 (se chunk ha il conv_tag o meno)
            has_all_conv_tag = bool(conv_tags and all(cnvtag in chunk_tags for cnvtag in conv_tags))

            # cos_sim con query, sempre necessaria
            raw_vector = getattr(rec, "vector", None)
            if isinstance(raw_vector, dict): # gestisce sia liste/array che dict come vettori
                chunk_vector = next(iter(raw_vector.values())) if raw_vector else None
            else:
                chunk_vector = raw_vector

            if chunk_vector is None:
                logger.warning("!> Impossibile reperire il vettore per '%s'. Chunk SALTATO.", chunk_id,)
                continue
            else:
                cos_score = max(0.0, cosine_sim(query_vector, chunk_vector))

            if cos_score < MIN_COSINE_THRESHOLD:
                continue

            neighbor_info = candidate_neighbors.get(chunk_id, {})
            
            # Moltiplicatore su base Distance Class
            dist_factor = 1.0
            if "DIST" in order_by_to_upper:
                dist_factor = float(neighbor_info.get("distance_factor", 1.0))

            tag_alteration = 1.0
            matched_all = []
            if "TAGS" in order_by_to_upper:
                matched_conv = list(set(conv_tags).intersection(set(chunk_tags)))
                matched_assigned = list(set(assigned_tags).intersection(set(chunk_tags)))
                matched_auto = list(set(auto_tags).intersection(set(chunk_tags)))
                matched_all = list(set(all_query_tags).intersection(set(chunk_tags)))

                bonus_conv = len(matched_conv) * TAG_BOOST_ASSIGNED_CONV
                bonus_assigned = len(matched_assigned) * TAG_BOOST_ASSIGNED_CONV
                bonus_auto = len(matched_auto) * TAG_BOOST_AUTO

                if not has_all_conv_tag: # categoria 3: contano i match parziali
                    total_bonus = bonus_conv + bonus_assigned + bonus_auto
                else: # categoria 2: tutti fanno perfect_match con il conv_tag
                    total_bonus = bonus_assigned + bonus_auto

                if total_bonus > 0:
                    tag_alteration = 1.0 + min(total_bonus, TAG_MAX_BOOST)

            # Moltiplicatore VICINANZA a query pre-determinata
            predq_alteration = 1.0
            if predq_id:
                predq_alteration = float(neighbor_info.get("predq_alteration", 1.0))
                
            # Punteggio finale
            raw_final_score = cos_score * dist_factor * tag_alteration * predq_alteration

            candidate_data = {
                "chunk_id": chunk_id,
                "point": rec,
                "initial_score": round(cos_score, 4),
                "dist_factor": round(dist_factor, 3),
                "tag_alteration": round(tag_alteration, 3),
                "predq_alteration": round(predq_alteration, 3),
                "final_score": round(raw_final_score, 4),
                "_raw_final_score": raw_final_score,
                "source_retrieval": "distance_retriever",
                "category": 2 if has_all_conv_tag else 3,
                "vector": chunk_vector,
                "payload": payload,
                "chunk_tags": chunk_tags
            }

            if has_all_conv_tag:
                category2_candidates.append(candidate_data)
            else:
                category3_candidates.append(candidate_data)


        # Riordinamento e unione categorie per return top_m
        category2_candidates.sort(key=lambda x: x["_raw_final_score"], reverse=True)
        category3_candidates.sort(key=lambda x: x["_raw_final_score"], reverse=True)

        prioritized_candidates = category2_candidates + category3_candidates
        for item in prioritized_candidates:
            del item["_raw_final_score"]

        return prioritized_candidates[:top_m]

EOF